<a href="https://colab.research.google.com/github/Winter-at/mid/blob/main/411322370_%E6%9C%9F%E4%B8%AD%E8%80%83B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. 系統目的

      提供系上同學透過自然語言（單一句話）完成飲品預約、取消與庫存查詢，並提供店長即時銷售統計與庫存管理的校園咖啡廳智慧系統。

2. 使用者會怎麼說
  
      1.我要一杯美式咖啡

      2.幫我取消剛剛訂的拿鐵兩杯

      3.請問現在拿鐵還剩多少？

      4.我想訂 2 杯鮮奶茶

      5.今天各品項賣得怎樣？
3. 資料表
| 表名 | 欄位 | 型態 | 說明 |
|---|---|---|---|
| menu | item_name | TEXT | 飲品名稱（主鍵） |
| menu | price | INTEGER | 單價 |
| menu | stock | INTEGER | 目前庫存數量 |
| orders | id | INTEGER | 訂單編號（主鍵、自動遞增） |
| orders | student_name | TEXT | 訂購學生姓名 |
| orders | item_name | TEXT | 訂購品項名稱（外鍵關聯 menu） |
| orders | quantity | INTEGER | 訂購數量 |
| orders | status | TEXT | 訂單狀態（'已完成')|

      初始資料：
      - menu 表：
        - `('美式咖啡', 60, 10)`
        - `('拿鐵', 80, 10)`
        - `('鮮奶茶', 70, 8)`
        - `('卡布奇諾', 85, 5)`
4. AI 的工作

- 輸入： 使用者的一句話

- 輸出格式： 動作,品項,數量

- 動作只能是： 訂購、取消、查詢、統計、無法判斷

- 品項只能是： 美式咖啡、拿鐵、鮮奶茶、卡布奇諾、無

- 數量的規則： 提取句中說明的數量（阿拉伯數字），若未特別提及數量則預設為 1；若為統計或無法判斷時輸出 0。

- 無法判斷時輸出： 無法判斷,無,0

- AI 不可以做的事：

  - 不得直接執行資料庫增刪修改或計算總金額。

  - 不得自行發明或回覆菜單外之品項。

  - 不得回覆除了指定格式 (動作,品項,數量) 以外的額外文字。
5. 功能與規則
| 功能 | 條件 | 系統回應 | 資料庫變化 |
|---|---|---|---|
| 訂購飲品 | 庫存足夠且單人當日累積未超過 3 杯（含本次），數量在 1~3 之間 | 回覆訂購成功訊息與總金額（例如：「訂購成功！拿鐵 2 杯，共 160 元。」） | `menu` 表扣減庫存，`orders` 表新增一筆 status 為 '已完成' 的記錄 |
| 訂購飲品 | 庫存不足 | 回覆拒絕訊息（例如：「訂購失敗：拿鐵庫存不足！」） | 無變化 |
| 訂購飲品 | 單人當日累積超過 3 杯上限 | 回覆拒絕訊息（例如：「訂購失敗：每人每日限購 3 杯，您已超出上限！」） | 無變化 |
| 訂購飲品 | 單次訂購數量不在 1~3 杯之間 | 回覆拒絕訊息（例如：「單次訂購數量必須介於 1 至 3 杯之間！」） | 無變化 |
| 取消訂購 | 該學生存在最新一筆 status 為 '已完成' 的該品項訂單 | 回覆取消成功訊息（例如：「已為您取消最新一筆拿鐵 1 杯的訂單。」） | `orders` 表最新該筆記錄 status 改為 '已取消'，`menu` 表補回相應庫存 |
| 取消訂購 | 查無該學生對應品項之已完成訂單 | 回覆提示訊息（例如：「取消失敗：查無您預約該品項的有效訂單。」） | 無變化 |
| 庫存查詢 | 指定品項存在於菜單中 | 回覆剩餘杯數（例如：「美式咖啡目前剩餘庫存：10 杯。」） | 無變化 |
| 銷售統計 | 使用者詢問銷售統計情況 | 列出所有品項之單價、初始庫存/目前庫存、已售杯數、營收明細與總營收 | 無變化 |
| 無法判斷 | 輸入內容無法解析為有效指令 | 回覆提示與現有菜單品項資訊 | 無變化 |
6. 例外情況

      1.輸入數量無效或超界： 使用者指定數量為 0、負數或超過 3 杯（例如：「我要訂 5 杯美式」），程式將直接拒絕並提示數量需在 1~3 杯之間。

      2.無名氏進行訂購/取消： 使用者執行「訂購」或「取消」時未提供姓名，系統提示需要填寫姓名才能辦理預約或取消。

      3.取消不存在或已取消的訂單： 同學重複發起取消，或從未訂購過該品項，系統查詢不到狀態為 '已完成' 的訂單記錄，回覆取消失敗提示。
7. 測試清單

      7-1 AI 解讀測試

      | # | 類型 | 使用者說的話 | 預期動作 | 預期品項 | 預期數量 |
      |---|---|---|---|---|---|
      | 1 | 正常 | 我要一杯美式咖啡 | 訂購 | 美式咖啡 | 1 |
      | 2 | 正常 | 幫我訂兩杯拿鐵 | 訂購 | 拿鐵 | 2 |
      | 3 | 正常 | 取消剛剛訂的鮮奶茶 | 取消 | 鮮奶茶 | 1 |
      | 4 | 正常 | 我要取消拿鐵 2 杯 | 取消 | 拿鐵 | 2 |
      | 5 | 正常 | 請問卡布奇諾還剩幾杯？ | 查詢 | 卡布奇諾 | 1 |
      | 6 | 正常 | 拿鐵還有嗎 | 查詢 | 拿鐵 | 1 |
      | 7 | 正常 | 今天各品項賣得怎樣 | 統計 | 無 | 0 |
      | 8 | 邊界 | 來三杯美式 | 訂購 | 美式咖啡 | 3 |
      | 9 | 異常 | 我想喝珍珠奶茶 | 無法判斷 | 無 | 0 |
      | 10 | 異常 | 今天天氣真好 | 無法判斷 | 無 | 0 |
      
      7-2 流程測試

      | # | 姓名 | 使用者說的話 | 預期結果 |
      |---|---|---|---|
      | 1 | 小明 | 我要兩杯拿鐵 | 訂購成功，金額 160 元（拿鐵庫存剩 8 杯） |
      | 2 | 小明 | 請問拿鐵還剩多少？ | 成功查詢，回覆拿鐵剩餘庫存：8 杯 |
      | 3 | 小明 | 我要一杯鮮奶茶 | 訂購成功，金額 70 元（小明今日累計已訂 3 杯） |
      | 4 | 小明 | 我要再一杯美式咖啡 | 訂購失敗，提示單人每日上限為 3 杯 |
      | 5 | 小明 | 幫我取消剛剛訂的拿鐵兩杯 | 取消成功，釋出 2 杯庫存（小明累計已訂杯數扣回至 1 杯，拿鐵庫存恢復至 10 杯） |
      | 6 | 小華 | 我要訂 4 杯美式咖啡 | 訂購失敗，提示單次訂購數量必須介於 1 至 3 杯之間 |
      | 7 | 小華 | 我要三杯卡布奇諾 | 訂購成功，金額 255 元（卡布奇諾庫存剩 2 杯） |
      | 8 | 小美 | 幫我取消美式咖啡 | 取消失敗，提示查無小美的美式咖啡訂單 |
      | 9 | 店長 | 今天各品項賣得怎樣？ | 成功列出全品項統計表（包含未賣出的美式咖啡，顯示銷量 0、剩餘 10；拿鐵銷量 0；鮮奶茶銷量 1；卡布奇諾銷量 3） |


In [17]:
# ============================================================
# 第 1 格：安裝套件、讀取金鑰、設定模型（老師提供，請勿修改）
# ============================================================
!pip install -q groq gradio

import os
import re
import time
import sqlite3
import pandas as pd
from IPython.display import display
from groq import Groq
import gradio as gr

# 判斷目前是在 Colab 還是本地端 Jupyter
try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# 🔴 金鑰名稱一律為 GROQ_API_KEY；絕對不可以把金鑰寫在程式裡，也不可以 print 出來
if IN_COLAB:
    try:
        GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    except Exception:
        print("❌ 讀不到金鑰：請到左側 🔑 新增名稱為 GROQ_API_KEY 的密鑰，並打開「筆記本存取權」")
        raise
else:
    GROQ_API_KEY = os.environ.get('GROQ_API_KEY')
    if not GROQ_API_KEY:
        raise RuntimeError("找不到環境變數 GROQ_API_KEY")

client = Groq(api_key=GROQ_API_KEY)

# 期中考指定模型；推理強度設為 low，可大幅減少 token 用量
# （唯一允許修改處：若 20b 額度用完，可改為 "openai/gpt-oss-120b"，並在規格書最後註明）
MODEL = "openai/gpt-oss-20b"
REASONING_EFFORT = "low"
print("✅ 環境：", "Colab" if IN_COLAB else "本地端", "｜模型：", MODEL)

✅ 環境： Colab ｜模型： openai/gpt-oss-20b


In [18]:
# (2) 建立資料庫與初始資料
# 每次執行時都會重新建立資料庫與資料表，並寫入初始的菜單資料

def init_db():
    conn = sqlite3.connect('cafe.db')
    cursor = conn.cursor()
    try:
        # 依照規格要求：每次執行先清除舊表，確保環境乾淨
        cursor.execute("DROP TABLE IF EXISTS orders;")
        cursor.execute("DROP TABLE IF EXISTS menu;")

        # 建立 menu 菜單資料表
        cursor.execute("""
            CREATE TABLE menu (
                item_name TEXT PRIMARY KEY,
                price INTEGER,
                stock INTEGER
            );
        """)

        # 建立 orders 訂單資料表
        cursor.execute("""
            CREATE TABLE orders (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                student_name TEXT,
                item_name TEXT,
                quantity INTEGER,
                status TEXT,
                FOREIGN KEY (item_name) REFERENCES menu (item_name)
            );
        """)

        # 放入初始菜單資料
        initial_menu = [
            ('美式咖啡', 60, 10),
            ('拿鐵', 80, 10),
            ('鮮奶茶', 70, 8),
            ('卡布奇諾', 85, 5)
        ]
        cursor.executemany("INSERT INTO menu (item_name, price, stock) VALUES (?, ?, ?);", initial_menu)

        conn.commit()
    finally:
        # 確保資料庫連線一定會被關閉
        conn.close()

# 執行初始化
init_db()
print("資料庫與初始菜單建立完成！")

資料庫與初始菜單建立完成！


In [19]:
# (3) 顯示資料表
# 用於查看目前 menu 與 orders 資料表中的內容

def show_tables():
    conn = sqlite3.connect('cafe.db')
    try:
        print("=== 菜單資料 (menu) ===")
        df_menu = pd.read_sql_query("SELECT * FROM menu;", conn)
        display(df_menu)

        print("\n=== 訂單資料 (orders) ===")
        df_orders = pd.read_sql_query("SELECT * FROM orders;", conn)
        display(df_orders)
    finally:
        conn.close()

# 執行查看
show_tables()

=== 菜單資料 (menu) ===


,item_name,price,stock
0,美式咖啡,60,10
1,拿鐵,80,10
2,鮮奶茶,70,8
3,卡布奇諾,85,5



=== 訂單資料 (orders) ===


,id,student_name,item_name,quantity,status


In [20]:
# (4) AI 解讀函式
# 負責呼叫 Groq 模型，將使用者的自然語言轉換成「動作,品項,數量」格式，並進行防呆與暫存

# 用於暫存 AI 回應的字典，避免重複呼叫相同的句子
ai_cache = {}

def parse_with_ai(user_input):
    # 如果這句話已經問過，直接從暫存讀取結果
    if user_input in ai_cache:
        return ai_cache[user_input]

    prompt = f"""
你是一個校園咖啡廳智慧系統的自然語言解析助理。
請將使用者的輸入解讀為：動作,品項,數量
規則：
1. 動作只能是：訂購、取消、查詢、統計、無法判斷 其中之一。
2. 品項只能是：美式咖啡、拿鐵、鮮奶茶、卡布奇諾、無 其中之一。
3. 數量的規則：提取句中說明的阿拉伯數字；若未特別提及數量則預設為 1；若動作為統計或無法判斷時輸出 0。
4. 如果無法判斷，請輸出：無法判斷,無,0
5. 絕對不要回覆除了指定格式 (動作,品項,數量) 以外的額外文字。

使用者輸入：「{user_input}」
"""

    response_text = "無法判斷,無,0"
    max_retries = 3

    for attempt in range(max_retries):
        try:
            completion = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                reasoning_effort=REASONING_EFFORT
            )
            response_text = completion.choices[0].message.content.strip()
            time.sleep(2) # 每次呼叫後暫停 2 秒
            break
        except Exception as e:
            # 檢查是否為 429 超過使用量錯誤
            if "429" in str(e):
                print(f"遇到 Rate Limit (429)，等待 30 秒後重試 (第 {attempt+1} 次)...")
                time.sleep(30)
            else:
                print(f"呼叫 AI 發生錯誤: {e}")
                time.sleep(2)
                break

    # 解析 AI 回傳的字串，進行嚴格的過濾與防呆檢查
    parts = response_text.split(',')

    # 預設值
    action = "無法判斷"
    item = "無"
    quantity = 0

    if len(parts) >= 1:
        raw_action = parts[0].strip()
        allowed_actions = ["訂購", "取消", "查詢", "統計", "無法判斷"]
        if raw_action in allowed_actions:
            action = raw_action

    if len(parts) >= 2:
        raw_item = parts[1].strip()
        allowed_items = ["美式咖啡", "拿鐵", "鮮奶茶", "卡布奇諾", "無"]
        if raw_item in allowed_items:
            item = raw_item

    # 用正規表示式從回覆中找出數字，找不到就當作 1（若動作非統計/無法判斷）
    found_nums = re.findall(r'\d+', response_text)
    if found_nums:
        quantity = int(found_nums[-1]) # 取最後一個出現的數字
    else:
        if action in ["訂購", "取消"]:
            quantity = 1
        else:
            quantity = 0

    # 若動作是統計或無法判斷，數量強制為 0
    if action in ["統計", "無法判斷"]:
        quantity = 0

    result = (action, item, quantity)
    ai_cache[user_input] = result # 存入暫存
    return result

In [21]:
# (5) AI 解讀測試
# 驗證 AI 解讀函式是否符合規格書中的 10 個測試案例

test_cases = [
    (1, "正常", "我要一杯美式咖啡", "訂購", "美式咖啡", 1),
    (2, "正常", "幫我訂兩杯拿鐵", "訂購", "拿鐵", 2),
    (3, "正常", "取消剛剛訂的鮮奶茶", "取消", "鮮奶茶", 1),
    (4, "正常", "我要取消拿鐵 2 杯", "取消", "拿鐵", 2),
    (5, "正常", "請問卡布奇諾還剩幾杯？", "查詢", "卡布奇諾", 1),
    (6, "正常", "拿鐵還有嗎", "查詢", "拿鐵", 1),
    (7, "正常", "今天各品項賣得怎樣", "統計", "無", 0),
    (8, "邊界", "來三杯美式", "訂購", "美式咖啡", 3),
    (9, "異常", "我想喝珍珠奶茶", "無法判斷", "無", 0),
    (10, "異常", "今天天氣真好", "無法判斷", "無", 0)
]

passed_count = 0

print("=== 開始 AI 解讀測試 ===")
for case_id, test_type, user_text, exp_action, exp_item, exp_qty in test_cases:
    # 取得 AI 原始回憶（為了觀察原始輸出，我們可以直接呼叫一次快取前的邏輯或顯示解析結果）
    pred_action, pred_item, pred_qty = parse_with_ai(user_text)

    is_match = (pred_action == exp_action and pred_item == exp_item and (exp_qty == "*" or pred_qty == exp_qty))
    if is_match:
        passed_count += 1
        status_str = "[通過]"
    else:
        status_str = "[失敗]"

    print(f"測試 #{case_id} ({test_type}) - {status_str}")
    print(f"  使用者說的話: {user_text}")
    print(f"  系統解讀結果: 動作={pred_action}, 品項={pred_item}, 數量={pred_qty}")
    print(f"  預期結果:     動作={exp_action}, 品項={exp_item}, 數量={exp_qty}\n")

print(f"測試結束！總共通過 {passed_count} / {len(test_cases)} 句。")

=== 開始 AI 解讀測試 ===
測試 #1 (正常) - [通過]
  使用者說的話: 我要一杯美式咖啡
  系統解讀結果: 動作=訂購, 品項=美式咖啡, 數量=1
  預期結果:     動作=訂購, 品項=美式咖啡, 數量=1

測試 #2 (正常) - [通過]
  使用者說的話: 幫我訂兩杯拿鐵
  系統解讀結果: 動作=訂購, 品項=拿鐵, 數量=2
  預期結果:     動作=訂購, 品項=拿鐵, 數量=2

測試 #3 (正常) - [通過]
  使用者說的話: 取消剛剛訂的鮮奶茶
  系統解讀結果: 動作=取消, 品項=鮮奶茶, 數量=1
  預期結果:     動作=取消, 品項=鮮奶茶, 數量=1

測試 #4 (正常) - [通過]
  使用者說的話: 我要取消拿鐵 2 杯
  系統解讀結果: 動作=取消, 品項=拿鐵, 數量=2
  預期結果:     動作=取消, 品項=拿鐵, 數量=2

測試 #5 (正常) - [通過]
  使用者說的話: 請問卡布奇諾還剩幾杯？
  系統解讀結果: 動作=查詢, 品項=卡布奇諾, 數量=1
  預期結果:     動作=查詢, 品項=卡布奇諾, 數量=1

測試 #6 (正常) - [通過]
  使用者說的話: 拿鐵還有嗎
  系統解讀結果: 動作=查詢, 品項=拿鐵, 數量=1
  預期結果:     動作=查詢, 品項=拿鐵, 數量=1

測試 #7 (正常) - [失敗]
  使用者說的話: 今天各品項賣得怎樣
  系統解讀結果: 動作=查詢, 品項=無, 數量=1
  預期結果:     動作=統計, 品項=無, 數量=0

測試 #8 (邊界) - [通過]
  使用者說的話: 來三杯美式
  系統解讀結果: 動作=訂購, 品項=美式咖啡, 數量=3
  預期結果:     動作=訂購, 品項=美式咖啡, 數量=3

測試 #9 (異常) - [通過]
  使用者說的話: 我想喝珍珠奶茶
  系統解讀結果: 動作=無法判斷, 品項=無, 數量=0
  預期結果:     動作=無法判斷, 品項=無, 數量=0

測試 #10 (異常) - [通過]
  使用者說的話: 今天天氣真好
  系統解讀結果: 動作=無法判斷, 品項=無, 數量=0
  預期結果:     動作=無法判斷,

In [22]:
# (6) 功能函式
# 包含處理訂購、取消、查詢、統計與無法判斷的商業邏輯，所有 SQL 皆使用 ? 參數防範隱患

def process_order(student_name, item_name, quantity):
    if not student_name or student_name.strip() == "":
        return "訂購失敗：請填寫您的姓名才能辦理預約！"

    if quantity < 1 or quantity > 3:
        return "訂購失敗：單次訂購數量必須介於 1 至 3 杯之間！"

    conn = sqlite3.connect('cafe.db')
    cursor = conn.cursor()
    try:
        # 1. 檢查品項是否存在並取得價格與庫存
        cursor.execute("SELECT price, stock FROM menu WHERE item_name = ?;", (item_name,))
        row = cursor.fetchone()
        if row is None:
            return f"訂購失敗：查無「{item_name}」此品項！"
        price, stock = row

        # 2. 檢查庫存是否充足
        if stock < quantity:
            return f"訂購失敗：{item_name}庫存不足！"

        # 3. 檢查單人當日累積是否超過 3 杯（含本次）
        # 簡單起見，計算該學生所有已完成訂單的總數量
        cursor.execute("""
            SELECT SUM(quantity) FROM orders
            WHERE student_name = ? AND status = '已完成';
        """, (student_name,))
        sum_row = cursor.fetchone()
        current_total = sum_row[0] if sum_row and sum_row[0] is not None else 0

        if current_total + quantity > 3:
            return "訂購失敗：每人每日限購 3 杯，您已超出上限！"

        # 4. 執行扣減庫存與新增訂單
        new_stock = stock - quantity
        cursor.execute("UPDATE menu SET stock = ? WHERE item_name = ?;", (new_stock, item_name))
        cursor.execute("""
            INSERT INTO orders (student_name, item_name, quantity, status)
            VALUES (?, ?, ?, '已完成');
        """, (student_name, item_name, quantity))

        conn.commit()
        total_price = price * quantity
        return f"訂購成功！{item_name} {quantity} 杯，共 {total_price} 元。（目前庫存剩 {new_stock} 杯）"
    finally:
        conn.close()

def process_cancel(student_name, item_name):
    if not student_name or student_name.strip() == "":
        return "取消失敗：請填寫您的姓名才能辦理取消！"

    conn = sqlite3.connect('cafe.db')
    cursor = conn.cursor()
    try:
        # 尋找該學生最新一筆 status 為 '已完成' 的該品項訂單
        cursor.execute("""
            SELECT id, quantity FROM orders
            WHERE student_name = ? AND item_name = ? AND status = '已完成'
            ORDER BY id DESC LIMIT 1;
        """, (student_name, item_name))
        row = cursor.fetchone()

        if row is None:
            return f"取消失敗：查無您預約該品項的有效訂單。"

        order_id, qty = row

        # 將訂單狀態改為 '已取消'
        cursor.execute("UPDATE orders SET status = '已取消' WHERE id = ?;", (order_id,))

        # 補回對應庫存
        cursor.execute("SELECT stock FROM menu WHERE item_name = ?;", (item_name,))
        menu_row = cursor.fetchone()
        if menu_row:
            current_stock = menu_row[0]
            new_stock = current_stock + qty
            cursor.execute("UPDATE menu SET stock = ? WHERE item_name = ?;", (new_stock, item_name))

        conn.commit()
        return f"已為您取消最新一筆 {item_name} {qty} 杯的訂單。（庫存已恢復）"
    finally:
        conn.close()

def process_query(item_name):
    conn = sqlite3.connect('cafe.db')
    cursor = conn.cursor()
    try:
        cursor.execute("SELECT stock FROM menu WHERE item_name = ?;", (item_name,))
        row = cursor.fetchone()
        if row is None:
            return f"查詢失敗：查無「{item_name}」此品項！"
        stock = row[0]
        return f"{item_name}目前剩餘庫存：{stock} 杯。"
    finally:
        conn.close()

def process_stats():
    conn = sqlite3.connect('cafe.db')
    cursor = conn.cursor()
    try:
        # 使用 JOIN 與 GROUP BY 統計各品項銷售情況
        query = """
            SELECT
                m.item_name,
                m.price,
                (m.stock + COALESCE(SUM(CASE WHEN o.status = '已完成' THEN o.quantity ELSE 0 END), 0)) AS initial_stock,
                m.stock AS current_stock,
                COALESCE(SUM(CASE WHEN o.status = '已完成' THEN o.quantity ELSE 0 END), 0) AS sold_qty,
                (COALESCE(SUM(CASE WHEN o.status = '已完成' THEN o.quantity ELSE 0 END), 0) * m.price) AS revenue
            FROM menu m
            LEFT JOIN orders o ON m.item_name = o.item_name
            GROUP BY m.item_name;
        """
        cursor.execute(query)
        rows = cursor.fetchall()

        result_str = "=== 今日各品項銷售統計表 ===\n"
        total_revenue = 0
        for row in rows:
            i_name, price, init_s, curr_s, sold_q, rev = row
            total_revenue += rev
            result_str += f"- 品項: {i_name} | 單價: {price} | 初始庫存: {init_s} | 目前庫存: {curr_s} | 已售杯數: {sold_q} | 營收: {rev} 元\n"
        result_str += f"---------------------------------\n總營收：{total_revenue} 元"
        return result_str
    finally:
        conn.close()

def process_unknown():
    conn = sqlite3.connect('cafe.db')
    cursor = conn.cursor()
    try:
        cursor.execute("SELECT item_name, price, stock FROM menu;")
        items = cursor.fetchall()
        menu_desc = ", ".join([f"{item[0]}({item[1]}元/剩{item[2]}杯)" for item in items])
        return f"無法理解您的指令。我們目前的菜單有：{menu_desc}。請重新輸入您的需求。"
    finally:
        conn.close()

In [23]:
# (7) 主流程與流程測試
# 整合 AI 解讀與功能函式，並執行規格書中的 9 個完整流程測試案例

def cafe_system_router(student_name, user_input):
    action, item, quantity = parse_with_ai(user_input)

    if action == "訂購":
        return process_order(student_name, item, quantity)
    elif action == "取消":
        return process_cancel(student_name, item)
    elif action == "查詢":
        return process_query(item)
    elif action == "統計":
        return process_stats()
    else:
        return process_unknown()

# 開始流程測試
flow_test_cases = [
    (1, "小明", "我要兩杯拿鐵"),
    (2, "小明", "請問拿鐵還剩多少？"),
    (3, "小明", "我要一杯鮮奶茶"),
    (4, "小明", "我要再一杯美式咖啡"),
    (5, "小明", "幫我取消剛剛訂的拿鐵兩杯"),
    (6, "小華", "我要訂 4 杯美式咖啡"),
    (7, "小華", "我要三杯卡布奇諾"),
    (8, "小美", "幫我取消美式咖啡"),
    (9, "店長", "今天各品項賣得怎樣？")
]

print("=== 開始執行流程測試 ===")
# 每次測試前先重新初始化資料庫，確保環境一致
init_db()

for case_id, name, text in flow_test_cases:
    print(f"\n[測試 #{case_id}] 姓名: {name} | 輸入: 「{text}」")
    response = cafe_system_router(name, text)
    print(f"系統回應:\n{response}")

print("\n=== 流程測試完畢 ===")

=== 開始執行流程測試 ===

[測試 #1] 姓名: 小明 | 輸入: 「我要兩杯拿鐵」
系統回應:
訂購成功！拿鐵 2 杯，共 160 元。（目前庫存剩 8 杯）

[測試 #2] 姓名: 小明 | 輸入: 「請問拿鐵還剩多少？」
系統回應:
拿鐵目前剩餘庫存：8 杯。

[測試 #3] 姓名: 小明 | 輸入: 「我要一杯鮮奶茶」
系統回應:
訂購成功！鮮奶茶 1 杯，共 70 元。（目前庫存剩 7 杯）

[測試 #4] 姓名: 小明 | 輸入: 「我要再一杯美式咖啡」
系統回應:
訂購失敗：每人每日限購 3 杯，您已超出上限！

[測試 #5] 姓名: 小明 | 輸入: 「幫我取消剛剛訂的拿鐵兩杯」
系統回應:
已為您取消最新一筆 拿鐵 2 杯的訂單。（庫存已恢復）

[測試 #6] 姓名: 小華 | 輸入: 「我要訂 4 杯美式咖啡」
系統回應:
訂購失敗：單次訂購數量必須介於 1 至 3 杯之間！

[測試 #7] 姓名: 小華 | 輸入: 「我要三杯卡布奇諾」
系統回應:
訂購成功！卡布奇諾 3 杯，共 255 元。（目前庫存剩 2 杯）

[測試 #8] 姓名: 小美 | 輸入: 「幫我取消美式咖啡」
系統回應:
取消失敗：查無您預約該品項的有效訂單。

[測試 #9] 姓名: 店長 | 輸入: 「今天各品項賣得怎樣？」
系統回應:
查詢失敗：查無「無」此品項！

=== 流程測試完畢 ===


In [24]:
# (8) Gradio 網頁
# 提供圖形化介面，讓使用者輸入姓名與訊息進行互動

def gradio_chat(student_name, user_input):
    if not student_name or student_name.strip() == "":
        return "請先輸入您的姓名或身分（例如：小明、店長）！"
    if not user_input or user_input.strip() == "":
        return "請輸入您想說的話。"
    return cafe_system_router(student_name.strip(), user_input.strip())

# 建立 Gradio 介面
demo = gr.Interface(
    fn=gradio_chat,
    inputs=[
        gr.Textbox(label="您的姓名 / 身分", placeholder="例如：小明 或 店長"),
        gr.Textbox(label="您的需求", placeholder="例如：我要兩杯拿鐵、今天各品項賣得怎樣？")
    ],
    outputs=gr.Textbox(label="系統回應"),
    title="校園咖啡廳智慧系統 ☕",
    description="歡迎使用校園咖啡廳智慧預約與查詢系統！請輸入您的名字與需求。"
)

# 啟動網頁（依照要求寫法）
if IN_COLAB:
    demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4560826558aa91f90d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
